## 1. Environment Building

In [1]:
import math
import numpy as np
import random

In [2]:
class ActionMapper:
    def __init__(self, board_size):
        self.board_size = board_size
        self.n_actions = board_size * board_size

    def idx_to_coord(self, action_idx):
        r = action_idx // self.board_size
        c = action_idx % self.board_size
        return (r, c)
        
    def coord_to_idx(self, action_coord):
        r, c = action_coord
        return r * self.board_size + c

In [3]:
class TicTacToeEnv:
    def __init__(self, board_size=3):
        self.board_size = board_size
        self.board = np.zeros((self.board_size, self.board_size), dtype=np.int8)
        self.player_1 = 1
        self.player_2 =-1
        self.current_player = self.player_1
        self.players = [self.player_1, self.player_2]

        self.action_mapper = ActionMapper(board_size)
        self.all_action_idx = list(range(self.action_mapper.n_actions))
        self.reward_points = {
            "draw": 0.5,
            "lose": -1,
            "win": 1,
            "invalid": -1,
            "continue": 0
        }

        self.state_dim = self.board_size * self.board_size #flatten board
        self.n_actions = self.action_mapper.n_actions
        self.n_states = 3**(self.board_size*self.board_size)
        
    def reset(self):
        self.board[:] = 0
        self.current_player = self.player_1
        return self.board.flatten()
        
    def is_draw(self):
        for i in range(self.board_size):
            for j in range(self.board_size):
                if self.board[i][j] == 0:
                    # found_empty_cell = True
                    return False
        return True

    def valid_actions(self):
        board = self.board.flatten()
        valid_actions = []
        for i in range(len(board)):
            if board[i] == 0:
                valid_actions.append(i)
        return valid_actions

    def random_valid_action(self):
        valid_actions = self.valid_actions()
        return random.choice(valid_actions)
    
    def check_winner(self):
        player_1_wins = self.board_size * self.player_1
        player_2_wins = self.board_size * self.player_2
        
        # -- diagonal search ---------
        downward_diag = []
        upward_diag = []
        for i in range(self.board_size):
            downward_diag.append(self.board[i][i])
            upward_diag.append(self.board[i][(self.board_size-1)-i])
        
        if sum(downward_diag) == player_1_wins or sum(upward_diag) == player_1_wins:
            return self.player_1
        elif sum(downward_diag) == player_2_wins or sum(upward_diag) == player_2_wins:
            return self.player_2
        # ---------------------------

        # ---- horizonal & vertical search ---------
        for i in range(self.board_size):
            row_cells_sum = self.board[i, :].sum()
            col_cells_sum = self.board[:, i].sum()
            if row_cells_sum == player_1_wins or col_cells_sum == player_1_wins:
                return self.player_1
            elif row_cells_sum == player_2_wins or col_cells_sum == player_2_wins:
                return self.player_2
        
        result = self.is_draw()
        if result:
            return "draw"
        return "continue"

    def get_current_player(self):
        return self.current_player

    def current_state(self):
        return self.board.flatten()
    
    def take_action(self, action_idx, player):
        """
        returns: [invalid, 1 <player_1>, -1 <player_2>, draw, None]
        """
        
        r, c = self.action_mapper.idx_to_coord(action_idx)
        if player not in self.players:
            raise Exception("invalid player")
        
        if self.board[r][c] != 0:
            return {"new_state": self.board.flatten(),
                    "reward": self.reward_points["invalid"],
                    "is_terminated": False}
        
        self.board[r][c] = player
        result = self.check_winner()
        
        game_over = result != "continue"

        if result == player:
            reward = self.reward_points["win"]
        elif result in self.players:
            reward = self.reward_points["lose"]
        else:
            reward = self.reward_points[result]

        self.current_player = -player #next player
        
        return {"new_state": self.board.flatten(),
                "reward": reward,
                "is_terminated": game_over}
        
    def render_board(self):
        symbols = {
            1: "X",
            -1: "O",
            0: " "
        }
    
        board = self.board.reshape(3,3)
    
        print("\n")
        for i in range(3):
            row = [symbols[v] for v in self.board[i]]
            print(" | ".join(row))
            if i < 2:
                print("--+---+--")
        print("\n")

In [4]:
game = TicTacToeEnv()
game.valid_actions()

[0, 1, 2, 3, 4, 5, 6, 7, 8]

In [5]:
game.render_board()



  |   |  
--+---+--
  |   |  
--+---+--
  |   |  




In [6]:
game.current_state()

array([0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8)

In [7]:
random.choice(game.valid_actions())

3

In [8]:
game.n_states

19683

# Q Network building

In [11]:
import random
import torch
from torch import nn 
from torch import optim
from collections import deque

In [12]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.ffn = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, state):
        return self.ffn(state)

In [13]:
class ReplayBuffer:
    def __init__(self, maxlen=5000):
        self.buffer = deque(maxlen=maxlen)
        
    def append(self, s, a, r, s_next, is_terminated):
        self.buffer.append((s, a, r, s_next, is_terminated))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, is_terminated = map(np.array, zip(*batch))
        return s, a, r, s_next, is_terminated

    def __len__(self):
        return len(self.buffer)

## Deep Q learning algorithm implementation

In [15]:
def epsilon_greedy_policy(q_net, state, env, epsilon=0.2):
    actions = env.all_action_idx 
    if random.random() < epsilon:
        return random.choice(actions) # <---- exploration
    else:
        state = torch.FloatTensor(state).unsqueeze(0) # batch X state_dim
        with torch.no_grad():
            q_values = q_net(state)  # batch X n_actions   
        return q_values.argmax(dim=1).item() # exploit

In [16]:
def get_approx_Qsa(q_net, state, action=None):
    state = torch.FloatTensor(state).unsqueeze(0)
    with torch.no_grad():
        q_values = q_net(state)
    if action is None:
        return q_values
    return q_values[action]

def get_max_Q_s(q_net, state, keepdim=True):
    state = torch.FloatTensor(state)
    with torch.no_grad():
        q_values = q_net(state)
    return q_values.max(dim=1, keepdim=keepdim)[0]

def get_argmax_Q_s(q_net, state):
    state = torch.FloatTensor(state)
    with torch.no_grad():
        q_values = q_net(state)
    return q_values.argmax(dim=1).item()

In [17]:
def get_policy_table(q_net, env):
    states = [State(s_idx, env.n_cols) for s_idx in range(env.n_states)]
    policy = np.zeros(env.n_states, dtype=np.int8)

    for state in states:
        state_feat_vec = state.get_state_feature_vec(env.n_states)
        state_feat_vec = torch.Tensor(state_feat_vec).unsqueeze(0) # 1 X feat_vec
        policy[state.idx] = get_argmax_Q_s(q_net, state_feat_vec)
    return policy


def get_Q_table(q_net, env):
    states = [State(s_idx, env.n_cols) for s_idx in range(env.n_states)]
    Q = np.zeros((env.n_states, env.n_actions), dtype=np.int8)
    for state in states:
        state_feat_vec = state.get_state_feature_vec(env.n_states)
        state_feat_vec = torch.Tensor(state_feat_vec).unsqueeze(0) # 1 X feat_vec
        q_values = get_approx_Qsa(q_net, state_feat_vec)
        Q[state.idx, :] = np.asarray(q_values)
    return Q

In [18]:
def train_step(env, online_net, target_net, optimizer, loss_fn,
               replay_buffer, batch_size=64, gamma=0.1):

    if len(replay_buffer) < batch_size:
        return

    s, a, r, next_s, is_terminated = replay_buffer.sample(batch_size)

    s = torch.Tensor(s) # batch X feat_vec
    a = torch.LongTensor(a).unsqueeze(1) # batch X 1
    r = torch.Tensor(r).unsqueeze(1) # batch X 1
    next_s = torch.Tensor(next_s) # batch X feat_vec

    is_terminated = torch.Tensor(is_terminated).unsqueeze(1)
    
    # Q(s, a; w_o) --->  q value of the corresponding action
    q_sa = online_net(s).gather(1, a)

    # Y_T =  r + γ max Q(s',a; W_T)
    with torch.no_grad():
        max_next_q = get_max_Q_s(target_net, next_s)
        y_T = r + gamma * max_next_q * (1 - is_terminated)

    # print("y T shape: ", y_T.shape)
    # print("q sa shape: ", q_sa.shape)
    
    loss = loss_fn(q_sa, y_T)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()    

In [19]:
def run_deep_q_learning(env, n_episodes=500, epsilon=0.7, 
                        epsilon_decay=0.01, gamma=0.99, lr=0.0001,
                        batch_size=64, target_update_freq=50):
    
    policy_histories = []
    Q_histories = []

    online_net = QNetwork(env.state_dim, env.n_actions)
    target_net = QNetwork(env.state_dim, env.n_actions)

    target_net.load_state_dict(online_net.state_dict())
    target_net.eval()
    optimizer = optim.Adam(online_net.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    replay_buffer = ReplayBuffer()

    rewards = []
    for episode_idx in range(n_episodes):
        env.reset()
        # --- initial state action pair -----
        state = env.current_state() #already in a feature vector form
        # -----------------------------------
        is_terminated = False
        episode_reward = 0
    
        # ----- epsilon decay -----------------------
        epsilon = max(0.05, epsilon - epsilon_decay)
        # -------------------------------------------
        
        while not is_terminated: # run iteration
            action = epsilon_greedy_policy(online_net,
                                           state,
                                           env,
                                           epsilon=epsilon)
            
            result = env.take_action(action, env.current_player)
            reward = result["reward"]
            next_state = result["new_state"]
            is_terminated = result["is_terminated"]

            # --- player 2 make a move ---------
            if not is_terminated:
                p_action = env.random_valid_action()
                result = env.take_action(p_action, env.current_player)
                next_state = result["new_state"]
                is_terminated = result["is_terminated"]
            # ----------------------------------
            
            replay_buffer.append(state,
                                 action,
                                 reward,
                                 next_state,
                                 is_terminated)

            train_step(
                    env,
                    online_net,
                    target_net, 
                    optimizer, 
                    loss_fn,
                    replay_buffer,
                    batch_size,
                    gamma)
 
            state = next_state
            episode_reward += reward
            
        rewards.append(episode_reward)

        if episode_idx % target_update_freq == 0:
            target_net.load_state_dict(online_net.state_dict())
            target_net.eval()
            
        if episode_idx % 500 == 0:
            avg = np.mean(rewards[-100:])
            print(f"Episode {episode_idx} | Avg Reward (100): {avg:.3f} | epsilon={epsilon}")
            
    return target_net

In [20]:
trained_q_net = run_deep_q_learning(game, n_episodes=100_000)

Episode 0 | Avg Reward (100): -2.000 | epsilon=0.69
Episode 500 | Avg Reward (100): -0.040 | epsilon=0.05
Episode 1000 | Avg Reward (100): 0.650 | epsilon=0.05
Episode 1500 | Avg Reward (100): 0.870 | epsilon=0.05
Episode 2000 | Avg Reward (100): 0.905 | epsilon=0.05
Episode 2500 | Avg Reward (100): 0.840 | epsilon=0.05
Episode 3000 | Avg Reward (100): 0.900 | epsilon=0.05
Episode 3500 | Avg Reward (100): 0.915 | epsilon=0.05
Episode 4000 | Avg Reward (100): 0.825 | epsilon=0.05
Episode 4500 | Avg Reward (100): 0.895 | epsilon=0.05
Episode 5000 | Avg Reward (100): 0.900 | epsilon=0.05
Episode 5500 | Avg Reward (100): 0.870 | epsilon=0.05
Episode 6000 | Avg Reward (100): 0.900 | epsilon=0.05
Episode 6500 | Avg Reward (100): 0.805 | epsilon=0.05
Episode 7000 | Avg Reward (100): 0.805 | epsilon=0.05
Episode 7500 | Avg Reward (100): 0.865 | epsilon=0.05
Episode 8000 | Avg Reward (100): 0.845 | epsilon=0.05
Episode 8500 | Avg Reward (100): 0.930 | epsilon=0.05
Episode 9000 | Avg Reward (100

In [21]:
game.current_state().reshape(3, 3)

array([[ 1,  1,  1],
       [-1, -1,  0],
       [ 1,  0, -1]], dtype=int8)

# PLAY WITH TRAINED AGENT

In [22]:
def agent_move(env, q_net):
    state = env.current_state()
    state_input = torch.FloatTensor(state).unsqueeze(0)

    with torch.no_grad():
        q_values = q_net(state_input).numpy()[0]

    valid = env.valid_actions()
    masked = np.full(env.n_actions, -1e9)
    masked[valid] = q_values[valid]
    return masked.argmax()

In [23]:
def human_move(env):
    valid = env.valid_actions()
    while True:
        action = int(input("Choose position (0-8): "))
        if action in valid:
            return action
        print("Invalid move. Try again.")

In [24]:
def play_game(env, agent):

    state = env.reset()

    print("\nPositions:")
    print("0 | 1 | 2")
    print("--+---+--")
    print("3 | 4 | 5")
    print("--+---+--")
    print("6 | 7 | 8")

    while True:

        env.render_board()

        player = env.get_current_player()

        if player == -1:
            action = human_move(env)
        else:
            action = agent_move(env, agent)
            print("Agent move:", action)

        result = env.take_action(action, player)

        if result["is_terminated"]:

            env.render_board()

            reward = result["reward"]

            if reward == -1:
                if player == 1:
                    print("You win!")
                else:
                    print("Agent wins!")

            elif reward == 1:
                if player == 1:
                    print("Agent wins!")
                else:
                    print("You win!")

            else:
                print("Draw!")

            break

In [ ]:
play_game(game, trained_q_net)


Positions:
0 | 1 | 2
--+---+--
3 | 4 | 5
--+---+--
6 | 7 | 8


  |   |  
--+---+--
  |   |  
--+---+--
  |   |  


Agent move: 2


  |   | X
--+---+--
  |   |  
--+---+--
  |   |  




# Save model

In [ ]:
torch.save(trained_q_net.state_dict(), "tictactoe_dqn_weights_v2.pth")

In [ ]:
# load saved model
agent = QNetwork(game.state_dim, game.n_actions)
agent.load_state_dict(torch.load("tictactoe_dqn_weights_v2.pth", weights_only=True))  # PyTorch 2.1+
agent.eval()